# Notebook 05 — Policy Comparison

Compare three inventory policies across 200 replications:
1. **Baseline**: Default (r, Q) parameters
2. **Aggressive**: Double r and Q
3. **Lean**: Halve r and Q

Cost model:
- Holding: ₹100 per part per day
- Stockout: ₹50,000 per aircraft per PAM day
- Labour: ₹2,000 per technician per shift

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.simulation import FleetSimulation
from src.inventory import INVENTORY_PARAMS

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120

HOURS_PER_DAY = 24
HOLDING_COST = 100       # ₹ per part per day
STOCKOUT_COST = 50000    # ₹ per aircraft per PAM day
LABOUR_COST = 2000       # ₹ per technician per shift

N_REPS = 200
SIM_DAYS = 365

print('Ready.')

## Define Policies

In [ ]:
def make_policy(multiplier):
    """Create inventory parameters with r and Q multiplied."""
    policy = {}
    for name, params in INVENTORY_PARAMS.items():
        policy[name] = {
            'reorder_point': max(1, int(params['reorder_point'] * multiplier)),
            'order_quantity': max(1, int(params['order_quantity'] * multiplier)),
            'initial_stock': max(1, int(params['initial_stock'] * multiplier)),
        }
    return policy

policies = {
    'Baseline': None,             # Default parameters
    'Aggressive': make_policy(2), # Double r and Q
    'Lean': make_policy(0.5),     # Halve r and Q
}

for name, pol in policies.items():
    if pol:
        print(f'{name}: engine r={pol["engine"]["reorder_point"]}, Q={pol["engine"]["order_quantity"]}')
    else:
        print(f'{name}: default parameters')

## Run Replications for Each Policy

In [ ]:
all_results = []

for policy_name, inv_params in policies.items():
    print(f'\nRunning {policy_name} policy...')
    sim = FleetSimulation(
        fleet_size=12,
        o_level_techs_day=8,
        i_level_techs_day=4,
        d_level_techs=6,
        sortie_interval_hours=48,
        inventory_params=inv_params,
    )
    
    for i in tqdm(range(N_REPS), desc=policy_name):
        result = sim.run(simulation_days=SIM_DAYS, random_seed=42 + i)
        
        # Compute costs
        # Holding cost: average stock * holding rate * days
        stock_df = pd.DataFrame(result['stock_history'])
        avg_stock = sum(
            stock_df[f'{s}_stock'].mean()
            for s in ['engine', 'avionics', 'hydraulics', 'airframe']
            if f'{s}_stock' in stock_df.columns
        )
        holding = avg_stock * HOLDING_COST * SIM_DAYS
        
        # Stockout cost
        pam_days = result['pam_fraction'] * 12 * SIM_DAYS
        stockout = pam_days * STOCKOUT_COST
        
        # Labour cost (8 O-level + 4 I-level + 6 D-level, 2 shifts/day)
        n_techs = 8 + 4 + 6
        labour = n_techs * 2 * LABOUR_COST * SIM_DAYS
        
        total_cost = holding + stockout + labour
        
        all_results.append({
            'policy': policy_name,
            'mcr': result['mcr'],
            'fmcr': result['fmcr'],
            'pam_fraction': result['pam_fraction'],
            'holding_cost': holding,
            'stockout_cost': stockout,
            'labour_cost': labour,
            'total_cost': total_cost,
        })

results_df = pd.DataFrame(all_results)
print('\nDone. Summary:')
results_df.groupby('policy')[['mcr', 'total_cost']].agg(['mean', 'std'])

## 1. MCR Box Plots by Policy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
policies_list = ['Lean', 'Baseline', 'Aggressive']
data = [results_df[results_df['policy'] == p]['mcr'] * 100 for p in policies_list]
bp = ax.boxplot(data, labels=policies_list, patch_artist=True)
colors_box = ['#e74c3c', '#3498db', '#2ecc71']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(y=75, color='red', linestyle='--', alpha=0.5, label='75% threshold')
ax.set_ylabel('MCR (%)', fontsize=12)
ax.set_title('MCR by Inventory Policy (200 replications)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Total Annual Cost Box Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data_cost = [results_df[results_df['policy'] == p]['total_cost'] / 1e6 for p in policies_list]
bp = ax.boxplot(data_cost, labels=policies_list, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Total Annual Cost (₹ millions)', fontsize=12)
ax.set_title('Total Annual Cost by Inventory Policy', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Cost-Effectiveness Scatter (Pareto)

In [ ]:
summary = results_df.groupby('policy').agg(
    mean_mcr=('mcr', 'mean'),
    mean_cost=('total_cost', 'mean'),
    std_mcr=('mcr', 'std'),
    std_cost=('total_cost', 'std'),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 6))
for idx, row in summary.iterrows():
    color = colors_box[['Lean', 'Baseline', 'Aggressive'].index(row['policy'])]
    ax.scatter(row['mean_cost'] / 1e6, row['mean_mcr'] * 100,
              s=200, c=color, zorder=5, edgecolors='black')
    ax.annotate(row['policy'], (row['mean_cost'] / 1e6, row['mean_mcr'] * 100),
               textcoords='offset points', xytext=(10, 5), fontsize=11)
    ax.errorbar(row['mean_cost'] / 1e6, row['mean_mcr'] * 100,
               xerr=row['std_cost'] / 1e6, yerr=row['std_mcr'] * 100,
               color=color, alpha=0.5, capsize=3)

ax.axhline(y=75, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Mean Annual Cost (₹ millions)', fontsize=12)
ax.set_ylabel('Mean MCR (%)', fontsize=12)
ax.set_title('Cost-Effectiveness: Pareto Analysis', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()